# Core Task Infrastructure
> Base classes and utilities for long-running MCP tasks

In [ ]:
#| default_exp tasks.core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Callable, Awaitable
from dataclasses import dataclass, field, asdict
from pathlib import Path
import time
import asyncio

from mcp.server.fastmcp import FastMCP

## Progress Tracking

In [ ]:
#| export
@dataclass
class TaskProgress:
    """Progress tracking for long-running tasks.
    
    Attributes
    ----------
    total_steps : int
        Total number of steps in the task.
    current_step : int
        Current step being executed (1-indexed).
    current_action : str
        Description of current action.
    started_at : float
        Unix timestamp when task started.
    artifacts : List[Dict[str, Any]]
        Intermediate results collected during execution.
    """
    total_steps: int
    current_step: int = 0
    current_action: str = ''
    started_at: float = field(default_factory=time.time)
    artifacts: List[Dict[str, Any]] = field(default_factory=list)
    
    @property
    def elapsed(self) -> float:
        """Seconds elapsed since task started."""
        return time.time() - self.started_at
    
    @property
    def percent_complete(self) -> float:
        """Percentage of task completed (0-100)."""
        if self.total_steps == 0:
            return 0.0
        return (self.current_step / self.total_steps) * 100
    
    def status_string(self) -> str:
        """Human-readable status string."""
        return f"[{self.current_step}/{self.total_steps}] {self.current_action}"

## Task Context

In [ ]:
#| export
class NbdevTaskContext:
    """Context manager for nbdev-specific long-running tasks.
    
    Wraps MCP task context with nbdev project awareness and
    progress tracking helpers.
    
    Parameters
    ----------
    project : Path
        Path to the nbdev project.
    total_steps : int
        Total number of steps in the task.
    task : Any, optional
        MCP ServerTaskContext for progress updates.
    
    Examples
    --------
    >>> ctx = NbdevTaskContext(project_path, total_steps=5)
    >>> await ctx.update(1, "Loading notebooks...")
    >>> await ctx.update(2, "Analyzing symbols...")
    """
    
    def __init__(self, project: Path, total_steps: int, task: Any = None):
        self.project = project
        self.task = task  # ServerTaskContext from MCP
        self.progress = TaskProgress(total_steps=total_steps)
    
    async def update(self, step: int, action: str) -> None:
        """Update progress and optionally send status to MCP.
        
        Parameters
        ----------
        step : int
            Current step number (1-indexed).
        action : str
            Description of current action.
        """
        self.progress.current_step = step
        self.progress.current_action = action
        
        if self.task is not None:
            try:
                await self.task.update_status(self.progress.status_string())
            except Exception:
                pass  # MCP update failed, continue anyway
    
    async def add_artifact(self, name: str, data: Any) -> None:
        """Store an intermediate result.
        
        Parameters
        ----------
        name : str
            Identifier for the artifact.
        data : Any
            The artifact data (should be JSON-serializable).
        """
        self.progress.artifacts.append({
            'name': name,
            'data': data,
            'timestamp': time.time()
        })
    
    def get_artifact(self, name: str) -> Optional[Any]:
        """Retrieve a stored artifact by name."""
        for artifact in reversed(self.progress.artifacts):
            if artifact['name'] == name:
                return artifact['data']
        return None
    
    def summary(self) -> Dict[str, Any]:
        """Get a summary of task execution."""
        return {
            'project': str(self.project),
            'total_steps': self.progress.total_steps,
            'completed_steps': self.progress.current_step,
            'elapsed_seconds': round(self.progress.elapsed, 2),
            'artifacts_count': len(self.progress.artifacts)
        }

## MCP Integration

In [ ]:
#| export
def enable_nbdev_tasks(mcp: FastMCP) -> None:
    """Enable experimental tasks on the MCP server.
    
    This must be called before registering any task-based tools.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    try:
        # MCP experimental tasks API
        if hasattr(mcp, 'server') and hasattr(mcp.server, 'experimental'):
            mcp.server.experimental.enable_tasks()
    except Exception:
        # Tasks not supported in this MCP version, continue without
        pass

## Task Helpers

In [ ]:
#| export
async def run_with_progress(
    steps: List[tuple[str, Callable[[], Awaitable[Any]]]],
    ctx: NbdevTaskContext
) -> List[Any]:
    """Run a sequence of async steps with progress updates.
    
    Parameters
    ----------
    steps : List[tuple[str, Callable]]
        List of (description, async_function) tuples.
    ctx : NbdevTaskContext
        Task context for progress updates.
    
    Returns
    -------
    List[Any]
        Results from each step.
    """
    results = []
    for i, (description, func) in enumerate(steps, 1):
        await ctx.update(i, description)
        result = await func()
        results.append(result)
        await ctx.add_artifact(f"step_{i}", {
            'description': description,
            'success': result is not None
        })
    return results

In [ ]:
#| export
def format_task_result(
    title: str,
    ctx: NbdevTaskContext,
    results: Dict[str, Any],
    errors: Optional[List[str]] = None
) -> Dict[str, Any]:
    """Format a task result with standard structure.
    
    Parameters
    ----------
    title : str
        Title for the result.
    ctx : NbdevTaskContext
        Task context with execution info.
    results : Dict[str, Any]
        Task-specific results.
    errors : List[str], optional
        List of error messages if any.
    
    Returns
    -------
    Dict[str, Any]
        Standardized result dictionary.
    """
    return {
        'ok': errors is None or len(errors) == 0,
        'title': title,
        'summary': ctx.summary(),
        'results': results,
        'errors': errors or []
    }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()